# Fig. 7h: manifold radius of the ground truth and of the sorted populations (evoked model)

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel h of the paper: manifold radius (average distance between the centre and the boundary of the manifold evoked by the 50 repetitions of a stimulus direction) of the ground truth neurons, of the sorted single-units of each spike sorter and of the multiunit threshold crossing. Source data and svg are saved in figures/8_source_data/fig7h/.

**method (as in the paper's notebook fig7_B_info_capacity_by_sorter_revised.ipynb)**
* the radius of each population is computed on 200 bootstraps of the neurons or units (population responses to the 360 stimulus directions, 200 latents, `neural_manifolds_replicaMFT`)
* the 200 bootstrapped radii of each population are read from the csv files of the paper (`igeometrics_by_sorter.csv`, `igeometrics_thresh_crossing.csv`); they are **not recomputed** here (the analysis needs the `mftma` package)
* box plots (notch, 200 bootstraps); red dashed line: mean radius of the ground truth
* comparison with the ground truth: two-sample t-test between the 200 bootstrapped radii (as in the paper's notebook)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)

# custom package
import seaborn as sns
from scipy.stats import ttest_ind

# paths (bootstrapped manifold capacities of the ground truth and of the sorted populations, evoked simulation, 1 hour)
NEURAL_CODE_PATH = "dataset/01_intermediate/neural_code/npx_evoked/"
IGEOM_SORT = NEURAL_CODE_PATH + "igeometrics_by_sorter.csv"
IGEOM_TC = NEURAL_CODE_PATH + "igeometrics_thresh_crossing.csv"

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the bootstrapped radii

In [2]:
%%time

igeom_df = pd.read_csv(IGEOM_SORT)
igeom_tc_df = pd.read_csv(IGEOM_TC)
igeom_df = pd.concat([igeom_df, igeom_tc_df[igeom_tc_df["Sorter"] == "thresh-crossing"]], axis=0, ignore_index=True)

# populations in the order of the figure
NAMES = {"ground_truth": "Ground\ntruth", "KS4": "KS4", "KS3": "KS3", "KS2.5": "KS2.5", "KS2": "KS2", "KS": "KS", "HS": "HS", "thresh-crossing": "TC"}
df = igeom_df[igeom_df["Sorter"].isin(NAMES)][["Sorter", "Radius"]].copy()
df["population"] = df["Sorter"].map(NAMES)
df["bootstrap"] = df.groupby("Sorter").cumcount()
df = df.set_index("Sorter").loc[list(NAMES)].reset_index(drop=True)[["population", "bootstrap", "Radius"]].rename(columns={"Radius": "manifold_radius"})
print(df.groupby("population", sort=False)["manifold_radius"].agg(["count", "median"]).round(3))

               count  median
population                  
Ground\ntruth    200   0.763
KS4              200   0.816
KS3              200   0.834
KS2.5            200   0.843
KS2              200   0.807
KS               200   0.739
HS               200   0.828
TC               200   0.687
CPU times: user 7.17 ms, sys: 21 µs, total: 7.19 ms
Wall time: 7.19 ms


## Radius of each sorted population against the ground truth

In [3]:
gt = df[df["population"] == "Ground\ntruth"]["manifold_radius"]
rows = []
for name in list(NAMES.values())[1:]:
    r = df[df["population"] == name]["manifold_radius"]
    t, p = ttest_ind(gt, r)
    rows.append(dict(population=name, median_radius=r.median(), change_percent=100 * (r.mean() - gt.mean()) / gt.mean(), t_statistic=t, p_value=p))
stats = pd.DataFrame(rows)
display(stats.round(4))
sorters = stats[stats["population"] != "TC"]
print("average change of the radius over the 6 spike sorters (%):", round(sorters["change_percent"].mean(), 1), "| without KS:", round(sorters[sorters["population"] != "KS"]["change_percent"].mean(), 1))

,population,median_radius,change_percent,t_statistic,p_value
0,KS4,0.8163,6.8226,-58.2528,0.0
1,KS3,0.8345,9.2889,-78.4028,0.0
2,KS2.5,0.8427,10.3792,-87.8616,0.0
3,KS2,0.8070,5.6180,-45.8033,0.0
4,KS,0.7387,-3.3426,18.2235,0.0
5,HS,0.8278,8.2662,-68.7285,0.0
6,TC,0.6869,-10.2004,79.3205,0.0


average change of the radius over the 6 spike sorters (%): 6.2 | without KS: 8.1


## Plot, same as the paper's panel h

In [4]:
# paper geometry (page 22 of the paper): canvas 110.25 x 92.57 pt, Arial 5 pt text, editable (src/nodes/validation/capacity_plots.py)
from src.nodes.validation.capacity_plots import plot_radius

def plot_source_data(df: pd.DataFrame):
    return plot_radius(df)

## Plot and save the figure and the source data

In [5]:
%%time

# plot
fig = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig7h", exist_ok=True)
plt.savefig("figures/8_source_data/fig7h/fig7h.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data (sheet 1: radius of the 200 bootstraps of each population; sheet 2: comparison with the ground truth)
with pd.ExcelWriter("figures/8_source_data/fig7h/fig7h.xlsx") as writer:
    df.to_excel(writer, sheet_name="radius_bootstraps", index=False)
    stats.to_excel(writer, sheet_name="vs_ground_truth", index=False)

## Check: replot from the saved source data

In [6]:
%%time

df = pd.read_excel("figures/8_source_data/fig7h/fig7h.xlsx", sheet_name="radius_bootstraps")
fig = plot_source_data(df)